In [1]:
# Install required Python ML analysis libraries
!pip install -q s3fs xgboost pyarrow scikit-learn


In [12]:
# Basic data loading and EDA
import pandas as pd
import numpy as np
cols = ['step', 'step_day', 'type', 'amount', 'oldbalanceorg', 'newbalanceorig',
        'oldbalancedest', 'newbalancedest', 'isfraud', 'isflaggedfraud',
        'txn_velocity', 'avg_amount_by_acct', 'amount_deviation']
df = pd.read_parquet('s3://mgmt59900-g3-paysim/curated/silver/', columns=cols)
print(df.shape)
print(df['isfraud'].value_counts(normalize=True))

(6362620, 13)
isfraud
0    0.998709
1    0.001291
Name: proportion, dtype: float64


In [13]:
# Check main features for training
df = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])]
print("Scoped shape:", df.shape)
print(df['isfraud'].value_counts(normalize=True))

Scoped shape: (2770409, 13)
isfraud
0    0.997035
1    0.002965
Name: proportion, dtype: float64


In [14]:
# Three-way split: train_only / validation / test
# train_only: steps 1-500 — used to fit model parameters
# val: steps 501-594 — used to tune hyperparameters (XGBoost)
# test: steps 595-743 — held out untouched until final evaluation
train_only = df[df['step'] <= 500]
val = df[(df['step'] > 500) & (df['step'] <= 594)]
test = df[df['step'] > 594]

for name, split in [('train_only', train_only), ('val', val), ('test', test)]:
    print(name, 'shape:', split.shape, 'fraud rate:', split['isfraud'].mean())


train_only shape: (2645779, 13) fraud rate: 0.0021018384377531154
val shape: (73344, 13) fraud rate: 0.013607111692844677
test shape: (51286, 13) fraud rate: 0.03225051671021331


In [15]:
# Feature engineering - Create features for model training
feature_cols = ['amount', 'txn_velocity', 'avg_amount_by_acct', 'amount_deviation']

X_train, y_train = train_only[feature_cols], train_only['isfraud']
X_val, y_val = val[feature_cols], val['isfraud']
X_test, y_test = test[feature_cols], test['isfraud']

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

Shapes: (2645779, 4) (73344, 4) (51286, 4)


In [25]:
# Compute a logistic regression model baseline
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(class_weight='balanced', max_iter=1000)
logreg.fit(X_train, y_train)
print("Baseline trained on scoped data with 4 features.")



Baseline trained on scoped data with 4 features.


In [17]:
#Train 4 XGBoost ML model variants with different tree depth/count/learning-rate combinations
#Evaluate each against the validation set (not test — test stays untouched) and select whichever 
#model variant scored highest on AUCPR.
import xgboost as xgb
from sklearn.metrics import average_precision_score

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

param_grid = [
    {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.1},
    {'max_depth': 5, 'n_estimators': 200, 'learning_rate': 0.1},
    {'max_depth': 5, 'n_estimators': 200, 'learning_rate': 0.05},
    {'max_depth': 7, 'n_estimators': 300, 'learning_rate': 0.05},
]

results = []
for params in param_grid:
    model = xgb.XGBClassifier(
        **params,
        scale_pos_weight=scale_pos_weight,
        eval_metric='aucpr',
        random_state=42
    )
    model.fit(X_train, y_train)
    val_probs = model.predict_proba(X_val)[:, 1]
    val_aucpr = average_precision_score(y_val, val_probs)
    results.append((params, val_aucpr, model))
    print(params, '-> val AUCPR:', round(val_aucpr, 4))

best_params, best_aucpr, best_xgb_model = max(results, key=lambda x: x[1])
print("\nBest params:", best_params, "with val AUCPR:", round(best_aucpr, 4))

scale_pos_weight: 474.7739615177126
{'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.1} -> val AUCPR: 0.0862
{'max_depth': 5, 'n_estimators': 200, 'learning_rate': 0.1} -> val AUCPR: 0.0858
{'max_depth': 5, 'n_estimators': 200, 'learning_rate': 0.05} -> val AUCPR: 0.0861
{'max_depth': 7, 'n_estimators': 300, 'learning_rate': 0.05} -> val AUCPR: 0.0851

Best params: {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.1} with val AUCPR: 0.0862


In [22]:
# Sanity leakage check
# Diagnostic evidence: confirms why oldbalanceorg/newbalanceorig were excluded above.
# Fraud shows oldbalanceorg == amount 97.93% of the time vs. 0.00% for legitimate
# transactions — a near-perfect leakage signal, mirroring the destination-balance
# artifact already found in the Data Quality Report. Excluded on both sides.
fraud_train = train_only[train_only['isfraud'] == 1]
legit_train = train_only[train_only['isfraud'] == 0]

fraud_drained = ((fraud_train['oldbalanceorg'] - fraud_train['amount']).abs() < 0.01).mean()
legit_drained = ((legit_train['oldbalanceorg'] - legit_train['amount']).abs() < 0.01).mean()

print("Fraud: oldbalanceorg == amount rate:", round(fraud_drained, 4))
print("Legit: oldbalanceorg == amount rate:", round(legit_drained, 4))

Fraud: oldbalanceorg == amount rate: 0.9793
Legit: oldbalanceorg == amount rate: 0.0


In [26]:
# Compute model metrics
from sklearn.metrics import precision_score, recall_score, average_precision_score

for name, model in [('LogReg', logreg), ('XGBoost (tuned)', best_xgb_model)]:
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    print(name,
          'precision:', round(precision_score(y_test, preds), 4),
          'recall:', round(recall_score(y_test, preds), 4),
          'AUCPR:', round(average_precision_score(y_test, probs), 4))

flagged = test['isflaggedfraud']
print('isFlaggedFraud baseline',
      'precision:', round(precision_score(y_test, flagged), 4),
      'recall:', round(recall_score(y_test, flagged), 4))

LogReg precision: 0.1583 recall: 0.4347 AUCPR: 0.2109
XGBoost (tuned) precision: 0.0926 recall: 0.5266 AUCPR: 0.1858
isFlaggedFraud baseline precision: 1.0 recall: 0.0048


In [27]:
# Apply the operational three-tier decision policy to the primary model's
# predicted probabilities on the held-out test set. Mirrors Module 6-05's
# threshold framework (0.00-0.30 approve, 0.30-0.70 review, 0.70-1.00 decline).
probs = best_xgb_model.predict_proba(X_test)[:, 1]

decision_tier = pd.cut(
    probs,
    bins=[0, 0.30, 0.70, 1.0],
    labels=['approve', 'review', 'decline'],
    include_lowest=True
)

tier_summary = pd.DataFrame({'decision_tier': decision_tier, 'is_fraud': y_test.values})
print(tier_summary['decision_tier'].value_counts())
print()
print(tier_summary.groupby('decision_tier', observed=True)['is_fraud'].agg(['count', 'sum', 'mean']))

decision_tier
review     40893
approve     7262
decline     3131
Name: count, dtype: int64

               count  sum      mean
decision_tier                      
approve         7262  100  0.013770
review         40893  927  0.022669
decline         3131  627  0.200256
